In [36]:
ch_name

'EEG LE-Pz'

In [35]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from scipy.signal import welch
from scipy.stats import zscore
import mne

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — edit these as needed
# ══════════════════════════════════════════════════════════════════════════════
FIF_PATH  = '/home/chengyi/Projects/Weimo/data_collection/annotated_fifs/chengyi_4_8_0.fif'
CH_INDEX  = 0        

JAW_EVENT_IDX  = 3   
JAW_PAD_BEFORE = 2.0 
JAW_PAD_AFTER  = 4.0 

MOT_PAD = 1.0        

OUT_FIG1 = 'fig1_stacked.png'
OUT_FIG2 = 'fig2_stacked.png'

BETA_BANDS = [
    (13, 16, 'β 13–16'),
    (16, 19, 'β 16–19'),
    (19, 22, 'β 19–22'),
    (22, 26, 'β 22–26'),
    (26, 30, 'β 26–30'),
]

WIN_SEC  = 0.5   
STEP_SEC = 0.1   

# Font Size Overrides (increased by ~4px)
FS_AXIS   = 12
FS_TICKS  = 11
FS_PIPE   = 10.5
FS_LEGEND = 11

# ── Colour palette ────────────────────────────────────────────────────────────
BG          = '#ffffff'
PANEL       = '#f6f8fa'
BORDER      = '#d0d7de'
TEXT        = '#1f2328'
MUTED       = '#57606a'
ACCENT      = '#0969da'
ORANGE      = '#cf222e'
GREEN       = '#1a7f37'
PURPLE      = '#8250df'
YELLOW      = '#9a6700'
BAND_COLORS = ['#0969da', '#1a7f37', '#cf222e', '#8250df', '#9a6700']


# ── Load ──────────────────────────────────────────────────────────────────────
raw = mne.io.read_raw_fif(FIF_PATH, preload=True, verbose=False)
sfreq = raw.info['sfreq']
eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])
ch_name   = raw.ch_names[eeg_picks[CH_INDEX]]


# ── Find annotation segments ──────────────────────────────────────────────────
jaw_onsets  = [(a['onset'], a['duration']) for a in raw.annotations if a['description'] == 'jaw_clench']
move_onsets = [(a['onset'], a['duration']) for a in raw.annotations if a['description'] == 'move']

jaw_onset, jaw_dur = jaw_onsets[JAW_EVENT_IDX]
JAW_T0 = jaw_onset - JAW_PAD_BEFORE
JAW_T1 = jaw_onset + JAW_PAD_AFTER

def segment_std(onset, dur):
    i0 = int(onset * sfreq)
    i1 = int((onset + dur) * sfreq)
    return raw.get_data()[eeg_picks[CH_INDEX], i0:i1].std()

best_onset, best_dur = min(move_onsets, key=lambda x: segment_std(*x))
MOT_T0 = best_onset - MOT_PAD
MOT_T1 = best_onset + best_dur + MOT_PAD


# ── Helpers ───────────────────────────────────────────────────────────────────
def get_segment(raw_obj, ch, t0, t1):
    idx = raw_obj.ch_names.index(ch)
    i0, i1 = int(t0 * sfreq), int(t1 * sfreq)
    times = np.arange(i1 - i0) / sfreq + t0
    return raw_obj.get_data()[idx, i0:i1], times

def preprocess(raw_in):
    r = raw_in.copy()
    r.notch_filter(60, picks='eeg', verbose=False)
    r.filter(13, 30, picks='eeg', verbose=False)
    r.set_eeg_reference('average', projection=False, verbose=False)
    return r

def style_ax(ax, ylabel, xlim):
    """Removed title logic and increased fontsizes."""
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=MUTED, labelsize=FS_TICKS)
    for spine in ax.spines.values():
        spine.set_edgecolor(BORDER)
    ax.set_ylabel(ylabel, color=MUTED, fontsize=FS_AXIS)
    ax.set_xlabel('Time (s)', color=MUTED, fontsize=FS_AXIS)
    ax.set_xlim(xlim)
    ax.grid(color=BORDER, lw=0.5, alpha=0.7)

def draw_pipeline(ax, steps, colors_list):
    """Increased font size for pipeline text labels."""
    ax.set_facecolor(BG)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    n, box_w, box_h = len(steps), 0.13, 0.52
    gap      = (1.0 - n * box_w) / (n + 1)
    y_center = 0.5
    for i, (label, col) in enumerate(zip(steps, colors_list)):
        x_left   = gap + i * (box_w + gap)
        x_center = x_left + box_w / 2
        if i > 0:
            ax.annotate('', xy=(x_left - 0.005, y_center),
                        xytext=(x_left - gap + 0.005, y_center),
                        arrowprops=dict(arrowstyle='->', color=MUTED, lw=1.2))
        ax.add_patch(FancyBboxPatch(
            (x_left, y_center - box_h / 2), box_w, box_h,
            boxstyle="round,pad=0.02",
            facecolor=col + '18', edgecolor=col, lw=1.2,
            transform=ax.transData, zorder=3))
        ax.text(x_center, y_center, label, ha='center', va='center',
                color=TEXT, fontsize=FS_PIPE, fontweight='bold', zorder=4, linespacing=1.3)

def shade_event(ax, onset, dur):
    ax.axvspan(onset, onset + dur, color=ORANGE, alpha=0.12, zorder=0)
    ax.axvline(onset,         color=ORANGE, lw=0.8, ls='--', alpha=0.6)
    ax.axvline(onset + dur,   color=ORANGE, lw=0.8, ls='--', alpha=0.6)

raw_proc = preprocess(raw)

# ══════════════════════════════════════════════════════════════════════════════
# Processing Loop
# ══════════════════════════════════════════════════════════════════════════════
configs = [
    dict(
        fig_id       = 'A',
        t0           = JAW_T0,
        t1           = JAW_T1,
        event_onset  = jaw_onset-0.50,
        event_dur    = jaw_dur+2.90,
        steps        = ['Notch\n60 Hz', 'Beta BP\n13–30 Hz', 'Avg\nRef', 'Log Mean\nBP'],
        pcols        = [ORANGE, GREEN, PURPLE, YELLOW],
        out          = OUT_FIG1,
    ),
    dict(
        fig_id       = 'B',
        t0           = MOT_T0,
        t1           = MOT_T1,
        event_onset  = best_onset,
        event_dur    = best_dur,
        steps        = ['Notch\n60 Hz', 'Beta BP\n13–30 Hz', 'Avg\nRef', 'Z-score\nNorm'],
        pcols        = [ORANGE, GREEN, PURPLE, ACCENT],
        out          = OUT_FIG2,
    ),
]

for cfg in configs:
    t0, t1 = cfg['t0'], cfg['t1']
    xlim   = (t0, t1)

    raw_seg,  times = get_segment(raw,      ch_name, t0, t1)
    proc_seg, _     = get_segment(raw_proc, ch_name, t0, t1)
    raw_seg  *= 1e6   
    proc_seg *= 1e6

    # Figure setup: Removed the title row from Gridspec
    fig = plt.figure(figsize=(14, 8), facecolor=BG)
    gs  = fig.add_gridspec(3, 1, height_ratios=[1, 0.28, 1],
                           hspace=0.4, left=0.08, right=0.96, top=0.95, bottom=0.08)

    # Raw waveform
    ax_raw = fig.add_subplot(gs[0])
    ax_raw.plot(times, raw_seg, color=ACCENT, lw=0.7, alpha=0.9)
    shade_event(ax_raw, cfg['event_onset'], cfg['event_dur'])
    style_ax(ax_raw, 'Amplitude (µV)', xlim)

    # Pipeline blocks
    ax_pipe = fig.add_subplot(gs[1])
    draw_pipeline(ax_pipe, cfg['steps'], cfg['pcols'])

    # Filtered waveform
    ax_filt = fig.add_subplot(gs[2])
    shade_event(ax_filt, cfg['event_onset'], cfg['event_dur'])

    if cfg['fig_id'] == 'A':
        win_samp  = int(WIN_SEC  * sfreq)
        step_samp = int(STEP_SEC * sfreq)
        win_centers, bp = [], {b[2]: [] for b in BETA_BANDS}
        i = 0
        while i + win_samp <= len(proc_seg):
            seg = proc_seg[i:i + win_samp]
            freqs, psd = welch(seg, fs=sfreq, nperseg=win_samp)
            for flo, fhi, label in BETA_BANDS:
                mask = (freqs >= flo) & (freqs <= fhi)
                bp[label].append(np.log10(np.mean(psd[mask]) + 1e-30))
            win_centers.append(times[i + win_samp // 2])
            i += step_samp
        win_centers = np.array(win_centers)
        for (flo, fhi, label), col in zip(BETA_BANDS, BAND_COLORS):
            ax_filt.plot(win_centers, bp[label], label=label, color=col, lw=1.6)
        style_ax(ax_filt, 'Log₁₀ Bandpower', xlim)
        ax_filt.legend(loc='upper right', fontsize=FS_LEGEND,
                       facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, framealpha=0.9)
    else:
        z_data = zscore(proc_seg)
        ax_filt.plot(times, z_data, color=ORANGE, lw=0.8, alpha=0.9)
        ax_filt.axhline( 0, color=MUTED,  lw=0.6, ls='--', alpha=0.6)
        ax_filt.axhline( 2, color=ORANGE, lw=0.6, ls=':',  alpha=0.5)
        ax_filt.axhline(-2, color=ORANGE, lw=0.6, ls=':',  alpha=0.5)
        style_ax(ax_filt, 'Z-score', xlim)

    fig.savefig(cfg['out'], dpi=150, facecolor=BG)
    plt.close(fig)
    print(f"Saved: {cfg['out']}")

/tmp/ipykernel_16829/560385528.py:57: RuntimeWarning: This filename (/home/chengyi/Projects/Weimo/data_collection/annotated_fifs/chengyi_4_8_0.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(FIF_PATH, preload=True, verbose=False)


Saved: fig1_stacked.png
Saved: fig2_stacked.png
